# Tech Layoff Count Prediction Using Machine Learning

## 1. Business Problem

The objective of this project is to predict the number of layoffs (layoffs_count) using company, workforce, AI adoption, financial, and hiring-related features.

This model can help organizations understand the factors contributing to workforce reductions and estimate potential layoffs.

## 2. Import Libraries

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Load Dataset

In [80]:
df = pd.read_csv("tech_layoffs_hiring_trends_elite_v2.csv")

## 4. Dataset Overview

In [81]:
df.head()

,record_id,company_name,industry,country,company_size,month,year,layoffs_count,layoff_percentage,reason_for_layoffs,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,top_hiring_role,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition
0,T0,Microsoft,AI,Singapore,Enterprise,Mar,2026,860,1.8,AI Automation,6.4,5.0,5426,Moderate Hiring,46.7,ML Engineer,-25.7,30.3,4.9,4.4,8.7,8.6,Bull Market
1,T1,Palantir,AI,Canada,Big Tech,Feb,2024,955,1.8,Cost Cutting,0.9,1.1,9666,Moderate Hiring,58.9,ML Engineer,-5.6,6.1,1.5,1.0,8.2,7.2,Bull Market
2,T2,Anthropic,Cybersecurity,USA,Mid-size,Apr,2025,18912,9.5,Overhiring Correction,7.1,3.9,437,Hiring Freeze,85.4,Frontend Developer,7.0,-23.6,-14.9,5.6,4.5,5.9,Recession
3,T3,Spotify,Gaming,USA,Mid-size,Jun,2025,18159,9.1,Cost Cutting,10.4,7.4,1075,Hiring Freeze,44.0,Frontend Developer,31.6,-22.3,-1.6,6.5,5.4,4.7,Recession
4,T4,Uber,Gaming,UK,Startup,Feb,2025,815,3.3,Market Slowdown,11.4,10.0,537,Moderate Hiring,53.2,Frontend Developer,85.3,26.6,9.8,9.3,6.7,5.8,Bull Market


In [82]:
df.shape

(12000, 23)

In [83]:
df.describe()

,year,layoffs_count,layoff_percentage,ai_automation_impact,ai_replacement_risk,open_roles,remote_jobs_percentage,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,2024.998750,5009.572083,12.780125,6.376983,7.208758,2884.395000,50.205150,22.466550,17.469175,5.920392,5.538750,6.496217,5.805033
std,0.817703,5159.360491,10.212612,3.544012,2.677148,2149.615907,23.299575,38.814405,24.359448,11.777264,2.583047,1.582356,1.649677
min,2024.000000,0.000000,0.000000,0.600000,1.000000,52.000000,10.000000,-45.000000,-25.000000,-28.000000,1.000000,1.300000,1.000000
25%,2024.000000,1369.500000,5.100000,3.500000,5.100000,1151.000000,29.800000,-11.200000,-3.500000,-2.700000,3.300000,5.400000,4.700000
50%,2025.000000,2733.000000,9.400000,5.900000,7.600000,2282.000000,50.500000,22.600000,17.500000,5.900000,5.600000,6.500000,5.900000
75%,2026.000000,6490.000000,17.300000,8.800000,10.000000,4432.250000,70.700000,55.700000,38.400000,14.600000,7.700000,7.600000,7.000000
max,2026.000000,19999.000000,40.000000,16.800000,10.000000,9997.000000,90.000000,90.000000,60.000000,38.200000,10.000000,10.000000,10.000000


In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   record_id               12000 non-null  object 
 1   company_name            12000 non-null  object 
 2   industry                12000 non-null  object 
 3   country                 12000 non-null  object 
 4   company_size            12000 non-null  object 
 5   month                   12000 non-null  object 
 6   year                    12000 non-null  int64  
 7   layoffs_count           12000 non-null  int64  
 8   layoff_percentage       12000 non-null  float64
 9   reason_for_layoffs      12000 non-null  object 
 10  ai_automation_impact    12000 non-null  float64
 11  ai_replacement_risk     12000 non-null  float64
 12  open_roles              12000 non-null  int64  
 13  hiring_trend            12000 non-null  object 
 14  remote_jobs_percentage  12000 non-null

In [85]:
df.columns

Index(['record_id', 'company_name', 'industry', 'country', 'company_size',
       'month', 'year', 'layoffs_count', 'layoff_percentage',
       'reason_for_layoffs', 'ai_automation_impact', 'ai_replacement_risk',
       'open_roles', 'hiring_trend', 'remote_jobs_percentage',
       'top_hiring_role', 'stock_growth_percent', 'revenue_growth_percent',
       'salary_budget_change', 'ai_adoption_level', 'employee_sentiment',
       'job_security_score', 'market_condition'],
      dtype='object')

## 5. Feature Selection

Target Variable:
- layoffs_count

Removed Features:
- record_id
- company_name
- layoff_percentage
- reason_for_layoffs

These features were removed because they are identifiers or introduce data leakage.

In [86]:
drop_cols = [
    'record_id',
    'company_name',
    'layoff_percentage',
    'reason_for_layoffs'
]

x = df.drop(columns=drop_cols + ['layoffs_count'])

y = df['layoffs_count']

## 6. Data Preprocessing – Encoding

Machine learning models require numerical input.

- Ordinal features were manually mapped.
- Nominal features were converted using One-Hot Encoding.

In [87]:
categorical_cols = x.select_dtypes(include='object').columns

categorical_cols

Index(['industry', 'country', 'company_size', 'month', 'hiring_trend',
       'top_hiring_role', 'market_condition'],
      dtype='object')

In [88]:
comapny_size_map = {
    'Startup':0,
    'Mid-size':1,
    'Enterprise':2,
    'Big Tech':3
}

hiring_trend_map = {
    'Downsizing':0,
    'Hiring Freeze':1,
    'Moderate Hiring':2,
    'Aggressive Hiring':3
}


market_condition_map = {
    'Recession':0,
    'Stable':1,
    'Bull Market':3
}

x['company_size'] = x['company_size'].map(comapny_size_map)
x['hiring_trend'] = x['hiring_trend'].map(hiring_trend_map)
x['market_condition'] = x['market_condition'].map(market_condition_map)

In [89]:
x.head()

,industry,country,company_size,month,year,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,top_hiring_role,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition
0,AI,Singapore,2,Mar,2026,6.4,5.0,5426,2,46.7,ML Engineer,-25.7,30.3,4.9,4.4,8.7,8.6,3
1,AI,Canada,3,Feb,2024,0.9,1.1,9666,2,58.9,ML Engineer,-5.6,6.1,1.5,1.0,8.2,7.2,3
2,Cybersecurity,USA,1,Apr,2025,7.1,3.9,437,1,85.4,Frontend Developer,7.0,-23.6,-14.9,5.6,4.5,5.9,0
3,Gaming,USA,1,Jun,2025,10.4,7.4,1075,1,44.0,Frontend Developer,31.6,-22.3,-1.6,6.5,5.4,4.7,0
4,Gaming,UK,0,Feb,2025,11.4,10.0,537,2,53.2,Frontend Developer,85.3,26.6,9.8,9.3,6.7,5.8,3


In [90]:
one_hot_cols = [
    'industry',
    'country',
    'month',
    'top_hiring_role'
]

x = pd.get_dummies(x, columns=one_hot_cols, drop_first=True)
x

,company_size,year,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition,industry_Cloud,industry_Cybersecurity,industry_E-Commerce,industry_FinTech,industry_Gaming,industry_Social Media,country_Germany,country_India,country_Singapore,country_UK,country_USA,month_Aug,month_Dec,month_Feb,month_Jan,month_Jul,month_Jun,month_Mar,month_May,month_Nov,month_Oct,month_Sep,top_hiring_role_Cybersecurity Analyst,top_hiring_role_Data Scientist,top_hiring_role_DevOps Engineer,top_hiring_role_Frontend Developer,top_hiring_role_ML Engineer,top_hiring_role_Product Manager,top_hiring_role_Software Engineer
0,2,2026,6.4,5.0,5426,2,46.7,-25.7,30.3,4.9,4.4,8.7,8.6,3,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False
1,3,2024,0.9,1.1,9666,2,58.9,-5.6,6.1,1.5,1.0,8.2,7.2,3,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,1,2025,7.1,3.9,437,1,85.4,7.0,-23.6,-14.9,5.6,4.5,5.9,0,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
3,1,2025,10.4,7.4,1075,1,44.0,31.6,-22.3,-1.6,6.5,5.4,4.7,0,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False
4,0,2025,11.4,10.0,537,2,53.2,85.3,26.6,9.8,9.3,6.7,5.8,3,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,2,2024,5.6,5.3,5364,2,15.8,18.5,-16.8,-0.8,4.0,6.7,5.1,3,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
11996,1,2026,3.4,5.7,3865,1,51.8,75.3,9.7,-1.7,2.3,7.7,6.2,1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
11997,3,2025,8.3,8.7,544,2,88.8,40.9,11.1,10.7,7.0,8.0,5.1,0,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False
11998,3,2025,9.1,10.0,2243,2,37.6,24.2,44.2,27.4,8.9,7.3,6.2,3,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False


## 7. Feature Engineering

New features are created to help the model capture relationships between existing variables and improve predictive performance.

In [91]:
x['ai_workforce_pressure'] = x['ai_adoption_level'] * x['ai_replacement_risk']

In [92]:
x['financial_health'] = x['revenue_growth_percent'] + x['salary_budget_change']

In [93]:
x['hiring_strength'] = x['open_roles'] * x['hiring_trend']

In [108]:
pd.set_option('display.max_columns',None)
x.head()

,company_size,year,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition,industry_Cloud,industry_Cybersecurity,industry_E-Commerce,industry_FinTech,industry_Gaming,industry_Social Media,country_Germany,country_India,country_Singapore,country_UK,country_USA,month_Aug,month_Dec,month_Feb,month_Jan,month_Jul,month_Jun,month_Mar,month_May,month_Nov,month_Oct,month_Sep,top_hiring_role_Cybersecurity Analyst,top_hiring_role_Data Scientist,top_hiring_role_DevOps Engineer,top_hiring_role_Frontend Developer,top_hiring_role_ML Engineer,top_hiring_role_Product Manager,top_hiring_role_Software Engineer,ai_workforce_pressure,financial_health,hiring_strength
0,2,2026,6.4,5.0,5426,2,46.7,-25.7,30.3,4.9,4.4,8.7,8.6,3,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,22.00,35.2,10852
1,3,2024,0.9,1.1,9666,2,58.9,-5.6,6.1,1.5,1.0,8.2,7.2,3,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,1.10,7.6,19332
2,1,2025,7.1,3.9,437,1,85.4,7.0,-23.6,-14.9,5.6,4.5,5.9,0,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,21.84,-38.5,437
3,1,2025,10.4,7.4,1075,1,44.0,31.6,-22.3,-1.6,6.5,5.4,4.7,0,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,48.10,-23.9,1075
4,0,2025,11.4,10.0,537,2,53.2,85.3,26.6,9.8,9.3,6.7,5.8,3,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,93.00,36.4,1074


## 8. Train-Test Split

In [104]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(9600, 46)
(2400, 46)
(9600,)
(2400,)


In [106]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

print(x_train_scaled.shape)
print(x_test_scaled.shape)

(9600, 46)
(2400, 46)


## 9. Feature Scaling

## 10. Model Building

## 11. Model Evaluation

## 12. Decision Tree

## 13. Random Forest

## 14. Model Comparison

## 15. Feature Importance

## 16. Business Insights

## 17. Conclusion

,company_size,year,ai_automation_impact,ai_replacement_risk,open_roles,hiring_trend,remote_jobs_percentage,stock_growth_percent,revenue_growth_percent,salary_budget_change,ai_adoption_level,employee_sentiment,job_security_score,market_condition,industry_Cloud,industry_Cybersecurity,industry_E-Commerce,industry_FinTech,industry_Gaming,industry_Social Media,country_Germany,country_India,country_Singapore,country_UK,country_USA,month_Aug,month_Dec,month_Feb,month_Jan,month_Jul,month_Jun,month_Mar,month_May,month_Nov,month_Oct,month_Sep,top_hiring_role_Cybersecurity Analyst,top_hiring_role_Data Scientist,top_hiring_role_DevOps Engineer,top_hiring_role_Frontend Developer,top_hiring_role_ML Engineer,top_hiring_role_Product Manager,top_hiring_role_Software Engineer,ai_workforce_pressure,financial_health,hiring_strength
0,2,2026,6.4,5.0,5426,2,46.7,-25.7,30.3,4.9,4.4,8.7,8.6,3,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,22.00,35.2,10852
1,3,2024,0.9,1.1,9666,2,58.9,-5.6,6.1,1.5,1.0,8.2,7.2,3,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,1.10,7.6,19332
2,1,2025,7.1,3.9,437,1,85.4,7.0,-23.6,-14.9,5.6,4.5,5.9,0,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,21.84,-38.5,437
3,1,2025,10.4,7.4,1075,1,44.0,31.6,-22.3,-1.6,6.5,5.4,4.7,0,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,48.10,-23.9,1075
4,0,2025,11.4,10.0,537,2,53.2,85.3,26.6,9.8,9.3,6.7,5.8,3,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,93.00,36.4,1074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,2,2024,5.6,5.3,5364,2,15.8,18.5,-16.8,-0.8,4.0,6.7,5.1,3,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,21.20,-17.6,10728
11996,1,2026,3.4,5.7,3865,1,51.8,75.3,9.7,-1.7,2.3,7.7,6.2,1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,13.11,8.0,3865
11997,3,2025,8.3,8.7,544,2,88.8,40.9,11.1,10.7,7.0,8.0,5.1,0,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,60.90,21.8,1088
11998,3,2025,9.1,10.0,2243,2,37.6,24.2,44.2,27.4,8.9,7.3,6.2,3,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False,89.00,71.6,4486
